# 05b — LIMU-BERT → NIO Frozen Ablation

**SIH PS 26168 — Intelligent Dead Reckoning — Phase 3**

Compares two NIO variants on the same train/val/test split:

| Model | IMU Input | LIMU-BERT |
|-------|-----------|----------|
| A: Raw NIO | Raw 6-DOF IMU → NIO TCN | Not used |
| B: LIMU-BERT NIO | Raw IMU → LIMU-BERT (frozen) → proj → NIO TCN | Frozen |

**This is an ablation experiment.** The result determines whether LIMU-BERT features improve downstream NIO accuracy.
- If B >> A: integrate LIMU-BERT into the production pipeline
- If B ≈ A or B < A: document the result, keep LIMU-BERT offline only

LIMU-BERT checkpoint required: `checkpoints/limu_bert/limu_bert_best.pt`  
Reference NIO checkpoint: `checkpoints/nio_fixed/nio_fixed_best.pt` (fixed uncertainty v2)

In [ ]:
import os, sys, json, time, datetime
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

if not torch.cuda.is_available():
    raise RuntimeError('[FATAL] CUDA not available. LIMU-BERT ablation requires GPU.')

device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

LIMU_CKPT  = PROJECT_ROOT / 'checkpoints' / 'limu_bert'     / 'limu_bert_best.pt'
NIO_CKPT   = PROJECT_ROOT / 'checkpoints' / 'nio_fixed'     / 'nio_fixed_best.pt'
ABLATION_CKPT_DIR = PROJECT_ROOT / 'checkpoints' / 'limu_bert_nio'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'limu_bert_ablation'
PLOTS_DIR   = PROJECT_ROOT / 'plots'   / 'limu_bert_ablation'
for d in [ABLATION_CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if not LIMU_CKPT.exists():
    raise FileNotFoundError(f'LIMU-BERT checkpoint not found: {LIMU_CKPT}\nRun notebook 05 (LIMU-BERT training) first.')
if not NIO_CKPT.exists():
    raise FileNotFoundError(f'Fixed NIO checkpoint not found: {NIO_CKPT}\nRun notebook 09 (NIO v2 training) first.')

print(f'LIMU-BERT checkpoint: {LIMU_CKPT} ({LIMU_CKPT.stat().st_size/1e6:.2f} MB)')
print(f'Fixed NIO checkpoint: {NIO_CKPT} ({NIO_CKPT.stat().st_size/1e6:.2f} MB)')

In [ ]:
from src.datasets.inertial_odometry_dataset import InertialOdometryDataset
from src.models.inertial_odometry import NeuralInertialOdometry
from src.models.limu_bert_nio import LIMUBERTNIOModel

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)

WINDOW_SIZE, TRAIN_STRIDE, VAL_STRIDE = 100, 20, 30
BATCH_SIZE = 64
NUM_EPOCHS = 20   # Ablation: 50 epochs is sufficient to see trend
LR = 5e-4

train_ds = InertialOdometryDataset(split='train', window_size=WINDOW_SIZE, stride=TRAIN_STRIDE)
val_ds   = InertialOdometryDataset(split='val',   window_size=WINDOW_SIZE, stride=VAL_STRIDE)
test_ds  = InertialOdometryDataset(split='test',  window_size=WINDOW_SIZE, stride=50)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,                num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=128,        shuffle=False,                num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} windows | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

# Model B: LIMU-BERT (frozen) + NIO
model_b = LIMUBERTNIOModel(
    limu_bert_checkpoint=str(LIMU_CKPT),
    tcn_channels=[64, 128, 256],
    dropout=0.1,
    vel_loss_weight=0.5,
    device=str(device)
).to(device)

# Only train non-frozen params (projection + NIO heads)
optimizer_b = torch.optim.AdamW(model_b.trainable_parameters(), lr=LR, weight_decay=1e-4)
scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=NUM_EPOCHS, eta_min=1e-5)
scaler_b    = torch.cuda.amp.GradScaler(enabled=True)

print(f'Model B (LIMU-BERT NIO) trainable params: {sum(p.numel() for p in model_b.trainable_parameters()):,}')

In [ ]:
# Train Model B
best_val_loss_b = float('inf')
best_ckpt_b = ABLATION_CKPT_DIR / 'limu_bert_nio_best.pt'
b_val_disp_rmses = []

print(f'Training LIMU-BERT NIO for {NUM_EPOCHS} epochs...')
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model_b.train()
    ep_loss, n = 0.0, 0
    for batch in train_loader:
        x, gd, gv = batch['imu'].to(device), batch['disp'].to(device), batch['vel'].to(device)
        optimizer_b.zero_grad()
        with torch.cuda.amp.autocast():
            pd, pv, plv = model_b(x)
            loss, _, _ = model_b.compute_loss(pd, pv, plv, gd, gv)
        if not (torch.isnan(loss) or torch.isinf(loss)):
            scaler_b.scale(loss).backward()
            scaler_b.unscale_(optimizer_b)
            nn.utils.clip_grad_norm_(model_b.trainable_parameters(), max_norm=1.0)
            scaler_b.step(optimizer_b)
            scaler_b.update()
            ep_loss += loss.item(); n += 1
    scheduler_b.step()

    model_b.eval()
    val_loss, disp_errs = 0.0, []
    with torch.no_grad():
        for batch in val_loader:
            x, gd, gv = batch['imu'].to(device), batch['disp'].to(device), batch['vel'].to(device)
            with torch.cuda.amp.autocast():
                pd, pv, plv = model_b(x)
                vl, _, _ = model_b.compute_loss(pd, pv, plv, gd, gv)
            if not (torch.isnan(vl) or torch.isinf(vl)):
                val_loss += vl.item()
                disp_errs.extend(torch.norm(pd - gd, dim=1).cpu().numpy())

    disp_rmse = float(np.sqrt(np.mean(np.array(disp_errs)**2))) if disp_errs else float('nan')
    b_val_disp_rmses.append(disp_rmse)

    if val_loss < best_val_loss_b:
        best_val_loss_b = val_loss
        torch.save({'model_state_dict': model_b.state_dict(), 'epoch': epoch,
                    'best_val_loss': best_val_loss_b, 'disp_rmse': disp_rmse,
                    'seed': SEED}, best_ckpt_b)
        star = ' ✓'
    else:
        star = ''

    if epoch % 10 == 0 or epoch == 1 or star:
        elapsed = (time.time() - t0) / 60
        print(f'Ep {epoch:3d}/{NUM_EPOCHS} | ValLoss={val_loss:.4f} DispRMSE={disp_rmse:.2f}m | {elapsed:.1f}min{star}')

print(f'Training done. Best val loss: {best_val_loss_b:.4f}')

In [ ]:
# Evaluate both models on Test (Driver A)
def evaluate_on_test(model, test_loader, label):
    model.eval()
    disp_errs, vel_errs = [], []
    import time as _time
    latencies = []
    with torch.no_grad():
        for batch in test_loader:
            x, gd, gv = batch['imu'].to(device), batch['disp'].to(device), batch['vel'].to(device)
            t_start = _time.perf_counter()
            with torch.cuda.amp.autocast():
                pd, pv, plv = model(x)
            t_end = _time.perf_counter()
            latencies.append((t_end - t_start) / len(x) * 1000)  # ms per sample
            disp_errs.extend(torch.norm(pd - gd, dim=1).cpu().numpy())
            vel_errs.extend(torch.abs(pv[:, 0] - gv[:, 0]).cpu().numpy())

    da = np.array(disp_errs)
    return {
        'label': label,
        'displacement_rmse_m': round(float(np.sqrt(np.mean(da**2))), 3),
        'displacement_mae_m':  round(float(np.mean(da)), 3),
        'displacement_p95_m':  round(float(np.percentile(da, 95)), 3),
        'velocity_rmse_mps':   round(float(np.sqrt(np.mean(np.array(vel_errs)**2))), 3),
        'inference_latency_ms_per_sample': round(float(np.median(latencies)), 4),
    }

# Load fixed NIO v2 for comparison
import yaml
with open(PROJECT_ROOT / 'configs' / 'training.yaml') as f:
    nio_cfg = yaml.safe_load(f)['inertial_odometry']

model_a = NeuralInertialOdometry(
    input_dim=6,
    tcn_channels=nio_cfg.get('tcn_channels', [64,128,256]),
    kernel_size=nio_cfg.get('tcn_kernel_size', 3),
    dropout=0.0
).to(device)
ckpt_a = torch.load(NIO_CKPT, map_location='cpu', weights_only=False)
model_a.load_state_dict(ckpt_a['model_state_dict'])

res_a = evaluate_on_test(model_a, test_loader, 'A_RawIMU_NIO_v2_Fixed')
res_b = evaluate_on_test(model_b, test_loader, 'B_LIMUBERT_Frozen_NIO')

print('\n=== LIMU-BERT ABLATION RESULTS (Driver A Test) ===')
print(f"{'Metric':<35} {'A: Raw NIO v2':>20} {'B: LIMU-BERT NIO':>20}")
print('-' * 75)
for key in ['displacement_rmse_m', 'displacement_mae_m', 'displacement_p95_m',
            'velocity_rmse_mps', 'inference_latency_ms_per_sample']:
    a_v, b_v = res_a.get(key, 'N/A'), res_b.get(key, 'N/A')
    print(f"{key:<35} {str(a_v):>20} {str(b_v):>20}")

# Verdict
improvement = res_a['displacement_rmse_m'] - res_b['displacement_rmse_m']
pct_improvement = improvement / res_a['displacement_rmse_m'] * 100
print(f'\nDisplacement RMSE improvement (A→B): {improvement:+.2f}m ({pct_improvement:+.1f}%)')
if improvement > 2.0:
    verdict = 'LIMU-BERT IMPROVES NIO → consider integrating'
elif improvement < -1.0:
    verdict = 'LIMU-BERT DEGRADES NIO → keep offline (latency penalty not justified)'
else:
    verdict = 'LIMU-BERT NEUTRAL → not worth the latency overhead'
print(f'Verdict: {verdict}')

ablation_results = {
    'experiment': 'limu_bert_frozen_nio_ablation',
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'session': 'S1', 'driver': 'Driver A (held-out test)',
    'model_A': res_a,
    'model_B': res_b,
    'displacement_rmse_improvement_m': round(float(improvement), 3),
    'displacement_rmse_improvement_pct': round(float(pct_improvement), 2),
    'verdict': verdict,
    'limu_bert_checkpoint': str(LIMU_CKPT.relative_to(PROJECT_ROOT)),
    'nio_reference_checkpoint': str(NIO_CKPT.relative_to(PROJECT_ROOT)),
    'limu_bert_nio_checkpoint': str(best_ckpt_b.relative_to(PROJECT_ROOT)),
}

out_path = RESULTS_DIR / 'ablation_results.json'
with open(out_path, 'w') as f:
    json.dump(ablation_results, f, indent=2)
print(f'\nAblation results saved: {out_path}')